In [ ]:
# Import necessary libraries
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import (
    load_split, add_rul, load_test_rul, last_cycle_per_unit,
    SENSORS, SETTINGS,
)

In [ ]:
# Load data
train = add_rul(load_split("train"))
test = load_split("test")
test_rul = load_test_rul()

print("train:", train.shape, "| engines:", train["unit"].nunique())
print("test: ", test.shape,  "| engines:", test["unit"].nunique())
print("test RUL labels:", len(test_rul))

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
test_rul.head()

In [ ]:
# Engine lifetimes
lifetime = train.groupby("unit")["cycle"].max()
print(lifetime.describe())

* Engines fail at different cycle counts (128 to 362).

In [ ]:
# Check for missing values
print(f'Training set has {int(train.isna().sum().sum())} missing data in total\n'
    f'Test set has {int(test.isna().sum().sum())} missing data in total')

`train` and `test` has no missing value in any column, hence no need for imputation.

In [ ]:
# Data integrity check.
# Check that no cycle skips
g = train.groupby("unit")["cycle"]
assert (g.max() == g.count()).all()

In [ ]:
# Check for test set
g1 = test.groupby("unit")["cycle"]
assert (g1.max() == g1.count()).all()

In [ ]:
# Check that the RUL labels line up with the test engines.
assert test_rul.index.equals(last_cycle_per_unit(test).index)

### Constant sensors
A constant sensor means the sensor reports the exact same value in every row, for every engine, at every point in life. A sensor can only help predict failure if its readings change as the engine degrades. A constant has no variation to learn from, so it carries zero information about RUL. Standardization also computes (x - mean) / std, which divides by zero and produces NaNs.

In [ ]:
n_unique = train[SENSORS].nunique().sort_values()
print(n_unique.head(8))

**Constant Sensors: s1, s5, s10, s16, s18, s19**. They all have a single distinct value across all 20,631 rows, so they carry no information about degradation and will be dropped.  
**s6** has two distinct values. It isn't constant, but it can report only one od two readings. It is also a likely candidate to drop

In [ ]:
train[SETTINGS].nunique().sort_values()

In [ ]:
print(train[["op1", "op2"]].describe().T[["min", "max", "std"]])
print(train["s6"].value_counts())